# DPO Pair Construction

Convert judge outputs into (chosen, rejected) preference pairs.


In [ ]:
import os
import json
from datasets import Dataset, DatasetDict

from dpo_helpers import (
    load_jsonl,
    split_positive_negative,
    construct_dpo_pairs,
    domain_split,
)


In [ ]:
import os
from pathlib import Path

# Resolve repo root
REPO_ROOT = Path(__file__).resolve().parents[1]

OUT_DIR = REPO_ROOT / "Sky_outputs"

GUIDE_PATH = OUT_DIR / "guide.jsonl"
GUIDE_REV_PATH = OUT_DIR / "guide_reverse.jsonl"
BEST_OF_N_PATH = OUT_DIR / "best_of_n.jsonl"

SAVE_DIR = REPO_ROOT / "Skydpo_dataset"
SAVE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
guide = load_jsonl(GUIDE_PATH)
guide_rev = load_jsonl(GUIDE_REV_PATH)
outputs = load_jsonl(BEST_OF_N_PATH)

print(len(guide), len(guide_rev), len(outputs))

In [ ]:
guide_split = split_positive_negative(guide)
guide_rev_split = split_positive_negative(guide_rev)
output_split = split_positive_negative(outputs)

In [ ]:
dpo_pairs = construct_dpo_pairs(
    guide_split=guide_split,
    guide_rev_split=guide_rev_split,
    output_split=output_split,
)

dataset = Dataset.from_dict(dpo_pairs)
print("Total DPO pairs:", len(dataset))

In [ ]:
split = dataset.train_test_split(test_size=0.1, seed=42)

dataset_dict = DatasetDict({
    "train": split["train"],
    "validation": split["test"],
})

In [ ]:
domain_splits = domain_split(dataset)

In [ ]:
dataset_dict.save_to_disk(SAVE_DIR)

meta_info = {
    "total_pairs": len(dataset),
    "domains": list(set(dataset["tag"])),
    "pair_types": list(set(dataset["pair_type"])),
}

with open(os.path.join(SAVE_DIR, "metadata.json"), "w") as f:
    json.dump(meta_info, f, indent=2)

for domain, split in domain_splits.items():
    path = os.path.join(SAVE_DIR, f"domain_{domain}")
    os.makedirs(path, exist_ok=True)
    split["train"].save_to_disk(os.path.join(path, "train"))
    split["test"].save_to_disk(os.path.join(path, "validation"))

In [ ]:
print("DPO dataset saved successfully")
print("Train samples:", len(dataset_dict["train"]))
print("Validation samples:", len(dataset_dict["validation"]))
print("Metadata saved at:", f"{SAVE_DIR}/metadata.json")